# Sesión 06 - Lab Reto: Reseñas de productos de la cadena de electrodomésticos (sin solución)

La cadena de electrodomésticos de las Sesiones 04 y 05 ahora registra las reseñas que sus clientes dejan en la app, por sucursal y por producto. Cada reseña trae dos estructuras anidadas distintas: `tags` (un array simple de textos, sin más estructura) y `respuestas` (un array de mensajes de la tienda o del cliente, con `autor`/`texto`/`timestamp`, la misma forma que `mensajes` en el laboratorio guiado). Algunas reseñas llegaron reingestadas: una de forma idéntica, y dos con una versión actualizada (cambio de `estado`, con una respuesta nueva agregada).

Datos de este laboratorio:
- `reto_resenas_productos.json`: reseñas anidadas, con los mismos dos casos de duplicado que el laboratorio guiado (una reingesta idéntica, dos versionadas por `ultima_actualizacion`).
- `catalogo_productos_lab_reto.csv`: tabla de referencia chica con el nombre y la categoría de cada producto, para el join del final.

Antes de correr este notebook, sube ambos archivos a `/Volumes/dbassociate/default/vol_landing/sesion_06/`.

## Verificación del entorno

In [ ]:
dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_06")

## Parte 1: Leer el JSON anidado con esquema explícito

Definí un `StructType` explícito para el archivo, igual que en el Lab 1 del laboratorio guiado: un struct para cada elemento de `respuestas` (`autor`/`texto`/`timestamp`), y un `ArrayType(StringType())` para `tags`, a diferencia de `respuestas`, que es un array de texto simple, no de structs.

In [ ]:
# TODO: define un StructType para "respuestas" (autor, texto, timestamp) -- mismo patron que "mensajes" del Lab 1
# schema_respuesta = StructType([...])

# TODO: define el StructType completo del registro de resena, incluyendo:
#   resena_id (string), producto_id (string), sucursal_id (int), cliente_nombre (string),
#   calificacion (int), fecha_resena (timestamp), ultima_actualizacion (timestamp), estado (string),
#   tags (ArrayType(StringType())), respuestas (ArrayType(schema_respuesta))
# schema_resena = StructType([...])

# TODO: lee "/Volumes/dbassociate/default/vol_landing/sesion_06/reto_resenas_productos.json" con ese schema
# df_crudo = spark.read.schema(schema_resena).json(...)

# TODO: agrega las columnas de auditoria (ingestion_timestamp, source_system, batch_id) y escribe
#       en dbassociate.bronze.resenas_productos

## Parte 2: Explorar los dos arrays sin aplanar

Antes de aplanar, contá cuántos `tags` y cuántas `respuestas` tiene cada reseña con `size()`. ¿Alguna reseña tiene el array `respuestas` vacío? Pensá qué pasaría con esa fila si la explotaras con `explode()` en la Parte 3.

In [ ]:
# TODO: para cada resena, conta size(tags) y size(respuestas)

# TODO: identifica si hay alguna resena con respuestas vacio (size == 0) -- son las que no tuvieron
#       ninguna respuesta de tienda ni de cliente

## Parte 3: Aplanar tags con explode() y respuestas con posexplode()

Aplicá `explode()` sobre `tags` (no importa el orden, son etiquetas sueltas) y `posexplode()` sobre `respuestas` (sí importa el orden: la posición 0 es la primera respuesta que recibió la reseña). Pensá si conviene hacerlo en el mismo DataFrame o en dos DataFrames separados: ¿tiene sentido una fila por combinación de tag y respuesta, o cada array pertenece a un análisis distinto?

In [ ]:
# TODO: aplana "tags" con explode() en un DataFrame (una fila por resena + tag)
# df_tags = ...

# TODO: aplana "respuestas" con posexplode() en otro DataFrame (una fila por resena + respuesta, con su posicion)
# df_respuestas = ...

# Pista: si combinas ambos explodes en el mismo DataFrame, el resultado es el producto cartesiano
#        de tags x respuestas para cada resena -- probablemente no es lo que buscas.

## Parte 4: Manipulación de columnas y filtros

Sobre `df_tags` (o el DataFrame que hayas elegido), aplicá al menos una manipulación de columna (agregar, dividir o renombrar) y un filtro con más de una condición. Por ejemplo: quedate solo con reseñas de calificación baja (1 o 2) que tengan el tag `producto_defectuoso`, para priorizar seguimiento de calidad.

In [ ]:
# TODO: aplica al menos una manipulacion de columnas (withColumn, withColumnRenamed o drop)

# TODO: aplica un filtro con multiples condiciones combinadas con & u |

## Parte 5: Deduplicar

`RES-5006` llegó reingestada de forma idéntica; `RES-5008` y `RES-5013` llegaron con una versión actualizada (cambio de `estado`, con una respuesta nueva agregada). Resolvé ambos casos sobre Bronze, antes de aplanar. Mismo criterio que el Lab 2 del laboratorio guiado.

In [ ]:
# TODO: usa dropDuplicates() (sin subset) para eliminar la reingesta identica de RES-5006
# df_sin_exactos = ...

# TODO: usa row_number() sobre una ventana particionada por resena_id, ordenada por
#       ultima_actualizacion descendente, para quedarte con la version mas reciente de
#       RES-5008 y RES-5013
# df_resenas_deduplicadas = ...

## Parte 6: Agregaciones de negocio

Con las reseñas ya deduplicadas y aplanadas, calculá al menos: calificación promedio por producto, cantidad de reseñas por sucursal, y el tag más frecuente entre las reseñas de calificación baja (1-2). Enriquecé el resultado con `catalogo_productos_lab_reto.csv` (join chico, buen candidato para `broadcast()`).

In [ ]:
# TODO: carga catalogo_productos_lab_reto.csv y unelo (con broadcast) a las resenas por producto_id

# TODO: calcula calificacion promedio por producto (mean)

# TODO: calcula cantidad de resenas por sucursal (count)

# TODO: identifica el tag mas frecuente entre las resenas con calificacion 1 o 2

## Preguntas de reflexión (sin respuesta única)

- ¿Por qué `RES-5006` se resolvió con `dropDuplicates()` sin `subset`, pero `RES-5008` y `RES-5013` necesitaron `row_number()` sobre una ventana?
- ¿Por qué `tags` se aplanó con `explode()` y `respuestas` con `posexplode()`? ¿Qué información se perdería si usaras `explode()` para `respuestas`?
- Si esta tabla de reseñas tuviera 50 millones de filas en vez de un puñado, ¿qué pasaría con el plan de ejecución del join con `catalogo_productos_lab_reto` si `autoBroadcastJoinThreshold` se dejara en su valor default? ¿Y si `catalogo_productos_lab_reto` creciera hasta superar ese umbral?

## Limpieza

In [ ]:
# TODO: elimina las tablas que hayas creado en este notebook, por ejemplo:
# spark.sql("DROP TABLE IF EXISTS dbassociate.bronze.resenas_productos")
# spark.sql("DROP TABLE IF EXISTS dbassociate.silver.resenas_productos_detalle")